<a href="https://colab.research.google.com/github/acapodanno/assistant-ai/blob/main/GreenThumb_Assistant_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GreenThumb Marketplace - AI Customer Support Assistant


Reference github: [assistant-ai](https://github.com/acapodanno/assistant-ai)

In [76]:
!rm -rf assistant-ai
!rm -rf evaluation
!rm -rf data
!git clone https://github.com/acapodanno/assistant-ai.git
!mv assistant-ai/data ./data
!mv assistant-ai/evaluation ./evaluation
!rm -rf assistant-ai
!rm -rf evaluation/eval_deepeval.py
!rm -rf evaluation/eval_ragas.py
!rm -rf evaluation/test_dataset.json
!ls -la data

Cloning into 'assistant-ai'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 132 (delta 24), reused 126 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 310.96 KiB | 4.09 MiB/s, done.
Resolving deltas: 100% (24/24), done.
total 44
drwxr-xr-x 3 root root  4096 Jun 29 08:54 .
drwxr-xr-x 1 root root  4096 Jun 29 08:54 ..
drwxr-xr-x 5 root root  4096 Jun 29 08:54 knowledge_base
-rw-r--r-- 1 root root 12606 Jun 29 08:54 orders.json
-rw-r--r-- 1 root root 14839 Jun 29 08:54 tickets.json


## 1. Installazione Dipendenze

In [77]:
!pip install -q langchain langchain-openai langchain-chroma chromadb deepeval ragas pandas loguru pydantic litellm datasets python-dotenv rank-bm25 fastapi uvicorn chainlit

## 2. Configurazione Chiave API OpenAI

In [78]:
import os
from google.colab import userdata


os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 2. Models

In [79]:
from pydantic import BaseModel, Field
from enum import Enum
from typing import Optional, List
from datetime import datetime

class IssueType(str, Enum):
    MANCATA_CONSEGNA = "MANCATA_CONSEGNA"
    PRODOTTO_DANNEGGIATO = "PRODOTTO_DANNEGGIATO"
    RESO = "RESO"
    RIMBORSO = "RIMBORSO"
    ALTRO = "ALTRO"

class TicketStatus(str, Enum):
    OPEN = "OPEN"
    IN_PROGRESS = "IN_PROGRESS"
    CLOSED = "CLOSED"

class CustomerInfo(BaseModel):
    name: str
    email: str
    phone: str

class OrderDetails(BaseModel):
    products: list[dict] = Field(default_factory=list)
    total: Optional[float] = None
    shipping_address: Optional[str] = None
    carrier: Optional[str] = None
    tracking_number: Optional[str] = None

class Ticket(BaseModel):
    ticket_id: int
    customer: CustomerInfo
    order_id: str
    issue_type: IssueType
    issue_description: str
    order_details: OrderDetails = Field(default_factory=OrderDetails)
    status: TicketStatus = TicketStatus.OPEN
    created_at: datetime = Field(default_factory=datetime.now)

class AgentResponse(BaseModel):
    answer: str = Field(description="La risposta finale per il cliente.")
    confidence: str = Field(default="high", description="Il livello di confidenza.")
    sources: list[str] = Field(default_factory=list, description="Lista delle fonti esatte")
    needs_human: bool = Field(default=False, description="True se serve supporto umano.")

class ChatRequest(BaseModel):
    message: str

# 3. Define RAG


*   Ingestion
*   Retriever
*   Rag Chain
*   Run Rag







# 3.1 Ingestion

In [80]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import os

LOADER_MAP = {
    ".md":   lambda path: TextLoader(file_path=path, encoding="utf-8"),
    ".txt":  lambda path: TextLoader(file_path=path, encoding="utf-8"),
    ".pdf":  lambda path: PyPDFLoader(file_path=path),
}

def retrieve_documents() -> list[Document]:
    kb_root = "./data/knowledge_base"
    documents = []
    for root, _dirs, files in os.walk(kb_root):
        for filename in files:
            filepath = os.path.join(root, filename)
            ext = os.path.splitext(filename)[1].lower()
            loader_factory = LOADER_MAP.get(ext)
            if loader_factory is None:
                continue
            loader = loader_factory(filepath)
            docs = loader.load()
            category = os.path.basename(root)
            for doc in docs:
                doc.metadata["file_type"] = ext.lstrip(".")
                doc.metadata["source"] = filepath
                doc.metadata["category"] = category
            documents.extend(docs)
    return documents

SEPARATORS_MAP = {
    "md":   ["\n# ", "\n## ", "\n### ", "\n\n", "\n", " ", ""],
    "pdf":  ["\n\n", "\n", ". ", " ", ""],
    "txt":  ["\n\n", "\n", " ", ""],
}

def chunk_documents(documents: list[Document]) -> list[Document]:
    chunked = []
    for doc in documents:
        file_type = doc.metadata.get("file_type", "txt")
        separators = SEPARATORS_MAP.get(file_type, ["\n\n", "\n", " ", ""])
        splitter = RecursiveCharacterTextSplitter(
            separators=separators,
            chunk_size=800,
            chunk_overlap=120,
            length_function=len,
        )
        chunked.extend(splitter.split_documents([doc]))
    return chunked

# 3.2 Retriever

In [81]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
import pickle
import os

BM25_PATH = "./chroma_db/bm25_index.pkl"

def build_vector_store(chunked_documents: list[Document]):
    import chromadb
    client = chromadb.PersistentClient(path="./chroma_db")
    try:
        client.delete_collection("greenthumb_kb")
    except Exception:
        pass

    vector_store = Chroma.from_documents(
        documents=chunked_documents,
        embedding=OpenAIEmbeddings(model="text-embedding-3-large"),
        persist_directory="./chroma_db",
        collection_name="greenthumb_kb",
    )
    return vector_store

def build_sparse_retriever(chunked_documents: list[Document]) -> BM25Retriever:
    retriever = BM25Retriever.from_documents(chunked_documents)
    retriever.k = 3
    os.makedirs("./chroma_db", exist_ok=True)
    with open(BM25_PATH, "wb") as f:
        pickle.dump(retriever, f)
    return retriever

def get_dense_retriever():
    vector_store = Chroma(
        persist_directory="./chroma_db",
        collection_name="greenthumb_kb",
        embedding_function=OpenAIEmbeddings(model="text-embedding-3-large"),
    )
    return vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={'k': 3, 'fetch_k': 10}
    )

def get_sparse_retriever() -> BM25Retriever:
    if not os.path.exists(BM25_PATH):
        raise FileNotFoundError(f"Indice BM25 non trovato in {BM25_PATH}. Eseguire l'ingestion.")
    with open(BM25_PATH, "rb") as f:
        retriever = pickle.load(f)
    return retriever

def get_hybrid_retriever(query: str, bm25_weight: float = 0.3) -> list[Document]:
    bm25 = get_sparse_retriever()
    dense = get_dense_retriever()
    bm25_results = bm25.invoke(query)
    dense_results = dense.invoke(query)

    scores = {}
    seen = {}
    for rank, doc in enumerate(bm25_results):
        key = doc.page_content[:120]
        scores[key] = scores.get(key, 0) + bm25_weight / (rank + 1)
        seen[key] = doc

    for rank, doc in enumerate(dense_results):
        key = doc.page_content[:120]
        scores[key] = scores.get(key, 0) + (1 - bm25_weight) / (rank + 1)
        seen[key] = doc

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [seen[k] for k, _ in ranked[:3]]

# 3.3 Rag Chain

In [82]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SYSTEM_PROMPT = (
    "Sei l'assistente clienti di GreenThumb Marketplace, specializzato in giardinaggio e prodotti per l'orto. "
    "Rispondi alle domande dei clienti utilizzando ESCLUSIVAMENTE il contesto fornito dalla knowledge base.\n\n"
    "REGOLE:\n"
    "1. Cita sempre il nome del documento sorgente tra parentesi quadre (es. [spedizioni.md]).\n"
    "2. Se la risposta non è presente nel contesto, rispondi: "
    "'Non ho trovato questa informazione nella nostra knowledge base. "
    "Ti consiglio di contattare il supporto a supporto@greenthumb.it'\n"
    "3. Sii conciso, chiaro e usa un tono cordiale e professionale.\n"
    "4. Per le domande su prodotti, includi sempre il prezzo se disponibile nel contesto.\n\n"
    "CONTESTO:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
])

def _format_docs(docs: list[Document]) -> str:
    parts = []
    for doc in docs:
        source = doc.metadata.get("source", "fonte sconosciuta")
        filename = source.split("/")[-1]
        parts.append(f"[{filename}]\n{doc.page_content.strip()}")
    return "\n\n---\n\n".join(parts)

def setup_rag_chain(retriever=None):
    if retriever is not None:
        chain = (
            {"context": retriever | _format_docs, "input": RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
        )
        return chain
    else:
        chain = prompt | llm | StrOutputParser()

        def invoke_with_docs(inputs: dict) -> dict:
            docs = inputs.get("context", [])
            context_str = _format_docs(docs)
            answer = chain.invoke({"input": inputs["input"], "context": context_str})
            return {"answer": answer, "context": docs}

        return invoke_with_docs

# 3.4 Run Rag

In [83]:
from loguru import logger

def run_rag(query: str) -> dict:
    """
    Esegue il retrieval ibrido e la RAG chain per rispondere a una query.
    Assicura di aver eseguito 'python src/rag/run_ingestion.py' prima.
    """
    try:
        retrieved_docs = get_hybrid_retriever(query=query)
        rag_chain = setup_rag_chain()
        result = rag_chain({"input": query, "context": retrieved_docs})
        return result
    except FileNotFoundError as e:
        logger.error(f"RAG database is not initialized. ({str(e)})")
        return {"answer": f"Error: RAG database is not initialized. ({str(e)})", "context": []}

# 3.5 Test Rag

In [84]:
question = "Quali sono le politiche di reso di GreenThumb?"
logger.info(f"Test question: {question}")
try:
  result = run_rag(question)
  logger.info(result.get("answer", ""))
except Exception as e:
  logger.error(f"Error: {e}")

2026-06-29 08:55:09.452 | INFO     | __main__:<cell line: 0>:2 - Test question: Quali sono le politiche di reso di GreenThumb?
2026-06-29 08:55:09.456 | ERROR    | __main__:run_rag:14 - RAG database is not initialized. (Indice BM25 non trovato in ./chroma_db/bm25_index.pkl. Eseguire l'ingestion.)
2026-06-29 08:55:09.461 | INFO     | __main__:<cell line: 0>:5 - Error: RAG database is not initialized. (Indice BM25 non trovato in ./chroma_db/bm25_index.pkl. Eseguire l'ingestion.)


# 4 Agent

# Define tools


1.   get_order_by_id: va a leggere su un file json e recupera ordine richiesto.
2.   create_ticket: creazione di un ticket segnalando la problematica
3.   rag_knowledge_base: recupera le informazioni in base agli .md passati al rag





In [85]:
import json
from loguru import logger


def get_order_by_id(order_id: str) -> dict | None:
    logger.info(f"Fetching order: {order_id}")
    try:
        with open("./data/orders.json", "r") as f:
            orders = json.load(f)
        for order in orders:
            if order["order_id"] == order_id:
                return order
        return None
    except Exception:
        return None

def create_ticket(customer: dict, order_id: str, issue_type: str, issue_description: str, order_details: dict | None = None) -> dict:
    logger.info(f"Creating ticket for {order_id}")
    try:
        with open("./data/tickets.json", "r") as f:
            tickets = json.load(f)
    except Exception:
        tickets = []

    try:
        ticket = Ticket(
            ticket_id=len(tickets) + 1,
            customer=CustomerInfo(**customer),
            order_id=order_id,
            issue_type=IssueType(issue_type),
            issue_description=issue_description,
            order_details=OrderDetails(**(order_details or {}))
        )
        ticket_dict = ticket.model_dump(mode="json")
        tickets.append(ticket_dict)
        with open("./data/tickets.json", "w") as f:
            json.dump(tickets, f, indent=4, ensure_ascii=False)
        return {"ticket_id": ticket.ticket_id, "status": ticket.status.value, "message": "Successo"}
    except Exception as e:
        return {"error": str(e)}

def rag_knowledge_base(query: str) -> dict:
    result = run_rag(query)
    sources = [doc.metadata.get("source", "").split("/")[-1] for doc in result.get("context", [])]
    return {"answer": result.get("answer", ""), "sources": list(dict.fromkeys(sources))}

AVAILABLE_TOOLS = {
    "get_order_by_id": get_order_by_id,
    "create_ticket": create_ticket,
    "rag_knowledge_base": rag_knowledge_base,
}

def execute_tool(name: str, arguments: dict) -> dict:
    if name not in AVAILABLE_TOOLS:
        return {"error": f"Tool '{name}' non trovato."}

    tool_func = AVAILABLE_TOOLS[name]
    try:
        return tool_func(**arguments)
    except Exception as e:
        return {"error": f"Errore nell'esecuzione del tool '{name}': {str(e)}"}
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order_by_id",
            "description": "Recupera i dettagli completi di un ordine tramite il suo ID. Usare sempre prima di creare un ticket per ottenere i dati aggiornati dell'ordine.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "L'ID numerico dell'ordine (es. '1001')"
                    }
                },
                "required": [
                    "order_id"
                ]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_ticket",
            "description": "Apre un ticket di supporto strutturato per escalation verso il team umano. Chiamare SOLO quando non si riesce a risolvere autonomamente il problema del cliente. Popolare tutti i campi con le informazioni disponibili.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer": {
                        "type": "object",
                        "description": "Oggetto cliente estratto dall'ordine tramite get_order_by_id",
                        "properties": {
                            "name": {
                                "type": "string",
                                "description": "Nome completo del cliente"
                            },
                            "email": {
                                "type": "string",
                                "description": "Email del cliente"
                            },
                            "phone": {
                                "type": "string",
                                "description": "Numero di telefono del cliente"
                            }
                        },
                        "required": [
                            "name",
                            "email",
                            "phone"
                        ]
                    },
                    "order_id": {
                        "type": "string",
                        "description": "ID dell'ordine coinvolto nel problema (es. '1001')"
                    },
                    "issue_type": {
                        "type": "string",
                        "enum": [
                            "MANCATA_CONSEGNA",
                            "PRODOTTO_DANNEGGIATO",
                            "RESO",
                            "RIMBORSO",
                            "ALTRO"
                        ],
                        "description": "Categoria del problema: MANCATA_CONSEGNA (pacco non arrivato), PRODOTTO_DANNEGGIATO (arrivato rotto), RESO (vuole restituire), RIMBORSO (vuole rimborso), ALTRO (altri problemi)"
                    },
                    "issue_description": {
                        "type": "string",
                        "description": "Descrizione concisa e precisa del problema segnalato dal cliente, inclusi dettagli rilevanti come tracking, date e indirizzo"
                    },
                    "order_details": {
                        "type": "object",
                        "description": "Oggetto con i dettagli dell'ordine recuperati da get_order_by_id: prodotti, importo, date, corriere, tracking",
                        "properties": {
                            "products": {
                                "type": "array",
                                "description": "Lista dei prodotti dell'ordine"
                            },
                            "total": {
                                "type": "number",
                                "description": "Totale dell'ordine in EUR"
                            },
                            "shipping_address": {
                                "type": "string",
                                "description": "Indirizzo di spedizione"
                            },
                            "carrier": {
                                "type": "string",
                                "description": "Corriere utilizzato (es. GLS)"
                            },
                            "tracking_number": {
                                "type": "string",
                                "description": "Numero di tracking del corriere"
                            }
                        }
                    }
                },
                "required": [
                    "customer",
                    "order_id",
                    "issue_type",
                    "issue_description"
                ]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "rag_knowledge_base",
            "description": "Recupera informazioni dalla knowledge base di GreenThumb Marketplace per rispondere a domande sui prodotti, politiche di reso, spedizioni, metodi di pagamento, ecc.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "La domanda del cliente sulla knowledge base"
                    }
                },
                "required": [
                    "query"
                ]
            }
        }
    }
]

# Define Agent

In [86]:
import asyncio
import json
from litellm import completion
from loguru import logger

class GreenThumbAgent:
    def __init__(self, model: str = "openai/gpt-4o", max_iterations: int = 5):
        self.model = model
        self.max_iterations = max_iterations
        self.tools = tools

    def run(self, user_message: str, history: list = None) -> str:
        messages = []
        if history:
            messages.extend(history)
        else:
            messages.append({
                "role": "system",
                "content": (
                    "Sei l'assistente clienti di GreenThumb Marketplace. Aiuta il cliente usando i tool disponibili.\n\n"
                    "## USO DELLA KNOWLEDGE BASE (RAG)\n"
                    "Per domande su: prodotti, prezzi, guide di giardinaggio, politiche di spedizione, reso, garanzia, pagamenti\n"
                    "→ chiama SEMPRE il tool 'rag_knowledge_base' con la query del cliente PRIMA di rispondere.\n"
                    "→ Se la risposta del RAG non è esatta ma contiene informazioni correlate, riferiscile comunque (es. caratteristiche cesoie).\n"
                    "→ Nel campo JSON 'sources', copia l'array 'sources' del RAG.\n\n"
                    "## REGOLE DI ESCALATION (TICKET)\n"
                    "Devi SEMPRE invocare 'create_ticket' e impostare 'needs_human: true' se:\n"
                    "1. Il cliente non ha ricevuto un ordine già CONSEGNATO.\n"
                    "2. Il cliente vuole cambiare l'indirizzo di un ordine già IN SPEDIZIONE. Comunica che non è possibile farlo, poi apri ticket ALTRO.\n"
                    "3. Il cliente vuole un RESO o RIMBORSO.\n"
                    "4. Il cliente segnala un PRODOTTO DANNEGGIATO.\n\n"
                    "## ISTRUZIONI TICKET (ORDINE OBBLIGATORIO)\n"
                    "STEP 1: Chiama get_order_by_id PRIMA di creare il ticket.\n"
                    "STEP 2: Chiama create_ticket con i dati dell'ordine.\n"
                )
            })
        messages.append({"role": "user", "content": user_message})
        return self.__react_human_loop__(messages)

    def __react_human_loop__(self, messages: list) -> str:
        for _ in range(self.max_iterations):
            response = completion(model=self.model, messages=messages, tools=self.tools)
            msg = response.choices[0].message
            if msg.tool_calls:
                messages.append(msg)
                for tool_call in msg.tool_calls:
                    name = tool_call.function.name
                    args = json.loads(tool_call.function.arguments)
                    res = execute_tool(name, args)
                    messages.append({"role": "tool", "tool_call_id": tool_call.id, "name": name, "content": json.dumps(res, ensure_ascii=False)})
            else:
                final = completion(model=self.model, messages=messages, response_format=AgentResponse)
                return final.choices[0].message.content
        return json.dumps({"answer": "Timeout", "sources": [], "needs_human": True})

# Memory Agent - Summaritation

In [87]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, AIMessage
from typing import Any
from litellm import completion
from pydantic import PrivateAttr
import tiktoken

from loguru import logger

class SummaryBufferHistory(InMemoryChatMessageHistory):
    llm: Any
    max_token_limit: int = 100
    _summary: str = PrivateAttr(default="")
    _enc: Any = PrivateAttr(default=None)

    def __init__(self, llm: Any, max_token_limit: int = 100, **kwargs: Any):
        super().__init__(llm=llm, max_token_limit=max_token_limit, **kwargs)
        object.__setattr__(self, "_enc", tiktoken.encoding_for_model("gpt-3.5-turbo"))
        object.__setattr__(self, "_summary", "")

    def _count_tokens(self) -> int:
        return sum(
            len(self._enc.encode(m.content)) + 4
            for m in self.messages
        )

    def add_message(self, message: BaseMessage) -> None:
        super().add_message(message)
        if self._count_tokens() > self.max_token_limit:
            self._summarize()

    def _summarize(self):
        old_summary = ""
        recent_messages = []
        if not self.messages:
            return
        for m in self.messages:
            if isinstance(m, SystemMessage) and "Riassunto" in m.content:
                old_summary = m.content
            else:
                recent_messages.append(m)

        history_text = "\n".join(
            f"{m.type}:{m.content}" for m in recent_messages
        )
        prompt = f"""Aggiorna il riassunto della conversazione di supporto clienti di GreenThumb Marketplace.
        Riassunto precedente:
        {old_summary}
        Nuovi messaggi di interazione:
        {history_text}

        REGOLE TASSATIVE PER IL NUOVO RIASSUNTO:
        1. Includi l'identità del cliente (Nome, Email o Telefono) se è stata specificata.
        2. Riassumi sinteticamente tutti i dettagli dell'ordine citati (es. ID dell'ordine, se è in spedizione, consegnato o in reso).
        3. Elenca brevemente i prodotti discussi (es. vanga, lavanda, bulbi) e il motivo del contatto (es. richiesta info, ritardo consegna, avvio reso).
        4. Specifica se è stato attivato il tool di escalation o aperto un ticket di supporto.
        5. Formula il riassunto in terza persona in modo fattuale, privo di convenevoli."""
        response = completion(
            model=self.llm,
            messages=[{"role": "user", "content": prompt}]
        )
        object.__setattr__(self, "_summary", response.choices[0].message.content)
        self.clear()
        if self._summary:
            self.messages.append(
                SystemMessage(content=f"Riassunto: {self._summary}")
            )
        logger.info(f"Summary updated: {self._summary}")

# --- Utility per la gestione sessioni (usate in FastAPI) ---
_session_store = {}

def get_session_history(session_id: str) -> SummaryBufferHistory:
    """Recupera o crea la memoria per una data sessione."""
    if session_id not in _session_store:
        _session_store[session_id] = SummaryBufferHistory(llm="gpt-4o-mini", max_token_limit=150)
    return _session_store[session_id]

def get_formatted_history(session_id: str) -> list[dict]:
    """Ritorna la storia dei messaggi formattata per LiteLLM."""
    memory = get_session_history(session_id)
    formatted_history = []
    for m in memory.messages:
        if isinstance(m, SystemMessage):
            formatted_history.append({"role": "system", "content": m.content})
        elif isinstance(m, HumanMessage):
            formatted_history.append({"role": "user", "content": m.content})
        elif isinstance(m, AIMessage):
            formatted_history.append({"role": "assistant", "content": m.content})
    return formatted_history

def add_to_history(session_id: str, message: str, role: str = "user"):
    """Aggiunge un messaggio alla memoria della sessione."""
    memory = get_session_history(session_id)
    if role == "user":
        memory.add_message(HumanMessage(content=message))
    elif role == "assistant":
        memory.add_message(AIMessage(content=message))

## 4. API (FastAPI) & Web UI (Chainlit) [`src/api`]

In [88]:
from fastapi import FastAPI

agent = GreenThumbAgent()
app = FastAPI()

@app.post("/chat/{session_id}")
def chat(session_id: str, chat_request: ChatRequest) -> AgentResponse:
    formatted_history = get_formatted_history(session_id)
    response_json_str = agent.run(user_message=chat_request.message, history=formatted_history)
    try:
        response_dict = json.loads(response_json_str)
        text_answer = response_dict.get("answer", response_json_str)
    except json.JSONDecodeError:
        response_dict = {"answer": response_json_str, "confidence": "low", "sources": [], "needs_human": True}
        text_answer = response_json_str
    add_to_history(session_id, chat_request.message, role="user")
    add_to_history(session_id, text_answer, role="assistant")
    return response_dict

In [89]:
import chainlit as cl
import uuid
import json
import asyncio
from dotenv import load_dotenv

load_dotenv()
agent = GreenThumbAgent()

@cl.on_chat_start
async def on_chat_start():
    session_id = str(uuid.uuid4())
    cl.user_session.set("session_id", session_id)
    await cl.Message(content="Benvenuto su GreenThumb! Sono il tuo assistente virtuale.").send()

@cl.on_message
async def on_message(message: cl.Message):
    session_id = cl.user_session.get("session_id")
    user_query = message.content
    msg = cl.Message(content="")
    await msg.send()
    formatted_history = get_formatted_history(session_id)
    response_json_str = await asyncio.to_thread(agent.run, user_message=user_query, history=formatted_history)
    try:
        response_dict = json.loads(response_json_str)
        text_answer = response_dict.get("answer", response_json_str)
        sources = response_dict.get("sources", [])
    except json.JSONDecodeError:
        text_answer = response_json_str
        sources = []
    display_text = text_answer
    if sources:
        display_text += "\n\n*(Fonti: " + ", ".join(sources) + ")*"
    add_to_history(session_id, user_query, role="user")
    add_to_history(session_id, text_answer, role="assistant")
    msg.content = display_text
    await msg.update()

## 5. Script di Valutazione Ragas & DeepEval [`evaluation`]

In [90]:
import json
import os
from loguru import logger
from datasets import Dataset

try:
    import sys
    import types
    _vertexai_stub = types.ModuleType("langchain_community.chat_models.vertexai")
    class _ChatVertexAI: pass
    _vertexai_stub.ChatVertexAI = _ChatVertexAI
    sys.modules.setdefault("langchain_community.chat_models.vertexai", _vertexai_stub)

    from ragas import evaluate as ragas_evaluate
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.metrics.collections import answer_relevancy, faithfulness, context_precision, context_recall
    RAGAS_AVAILABLE = True
except Exception as e:
    logger.warning(f"RAGAS non disponibile: {e}")
    RAGAS_AVAILABLE = False
def run_ingestion():
    logger.info("Loading documents...")
    docs = retrieve_documents()
    logger.info(f"-> {len(docs)} documents loaded.")
    chunks = chunk_documents(docs)
    logger.info(f"-> {len(chunks)} chunks.")
    build_vector_store(chunks)
    build_sparse_retriever(chunks)
    logger.info("Ingestion completed!")

def run_ragas():
    if not RAGAS_AVAILABLE: return
    run_ingestion()
    with open("evaluation/test_dataset_rag.json", "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
    for item in eval_data:
        query = item["query"]
        res = run_rag(query)
        ragas_data["question"].append(query)
        ragas_data["answer"].append(res.get("answer", ""))
        ragas_data["contexts"].append([doc.page_content for doc in res.get("context", [])])
        ragas_data["ground_truth"].append(item["ground_truth"])

    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    rl = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
    re = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

    res_eval = ragas_evaluate(
        Dataset.from_dict(ragas_data),
        metrics=[answer_relevancy(), faithfulness(), context_precision(), context_recall()],
        llm=rl, embeddings=re
    )
    logger.info(f"Risultati Ragas: {res_eval}")

In [91]:
run_ragas()

2026-06-29 08:55:09.980 | INFO     | __main__:run_ingestion:24 - Loading documents...
2026-06-29 08:55:09.987 | INFO     | __main__:run_ingestion:26 - -> 30 documents loaded.
2026-06-29 08:55:09.996 | INFO     | __main__:run_ingestion:28 - -> 30 chunks.
2026-06-29 08:55:13.750 | INFO     | __main__:run_ingestion:31 - Ingestion completed!
/tmp/ipykernel_4480/3249690127.py:48: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  rl = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
/tmp/ipykernel_4480/3249690127.py:49: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import Ope

TypeError: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]

In [92]:
#!/usr/bin/env python3
import json
from loguru import logger
from deepeval.metrics import AnswerRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval import evaluate as deepeval_evaluate
from deepeval.evaluate.configs import AsyncConfig, CacheConfig

def run_deepeval():
    run_ingestion()
    with open("evaluation/test_dataset_agent.json", "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    agent = GreenThumbAgent()
    test_cases = []
    for item in eval_data:
        query = item["query"]
        raw_res = agent.run(query)
        try:
            res_obj = json.loads(raw_res)
            act_out = res_obj.get("answer", raw_res)
        except Exception:
            act_out = str(raw_res)
        test_cases.append(LLMTestCase(input=query, actual_output=act_out, expected_output=item["ground_truth"]))

    answer_relevancy = AnswerRelevancyMetric(threshold=0.5, model="gpt-4o")
    correctness = GEval(
        name="Agent Correctness",
        criteria="Valuta se la risposta rispetta le indicazioni (strumenti usati, escalation, tono).",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
        threshold=0.5, model="gpt-4o"
    )
    deepeval_evaluate(test_cases, [answer_relevancy, correctness], async_config=AsyncConfig(run_async=True, max_concurrent=1, throttle_value=10))

/tmp/ipykernel_4480/3840690636.py:5: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [ ]:
run_deepeval()

2026-06-29 08:57:56.782 | INFO     | __main__:run_ingestion:24 - Loading documents...
2026-06-29 08:57:56.805 | INFO     | __main__:run_ingestion:26 - -> 30 documents loaded.
2026-06-29 08:57:56.827 | INFO     | __main__:run_ingestion:28 - -> 30 chunks.
2026-06-29 08:57:58.062 | INFO     | __main__:run_ingestion:31 - Ingestion completed!
2026-06-29 08:57:58.820 | INFO     | __main__:get_order_by_id:6 - Fetching order: 1001
2026-06-29 08:58:00.949 | INFO     | __main__:create_ticket:18 - Creating ticket for 1001
2026-06-29 08:58:04.419 | INFO     | __main__:get_order_by_id:6 - Fetching order: 1004
2026-06-29 08:58:07.748 | INFO     | __main__:create_ticket:18 - Creating ticket for 1004
2026-06-29 08:58:12.102 | INFO     | __main__:get_order_by_id:6 - Fetching order: 1005
2026-06-29 08:58:13.904 | INFO     | __main__:create_ticket:18 - Creating ticket for 1005
2026-06-29 08:58:18.103 | INFO     | __main__:get_order_by_id:6 - Fetching order: 1002
2026-06-29 08:58:20.080 | INFO     | __mai